# 04 Filter fund universe

目标：生成pass/unknown/explicit_fail并以至少一个pass确定WFICN月资格。

- 输入：03 shareclass month panel；fund_filter.yaml
- 输出：shareclass_eligibility、eligible_wficn_months及原因审计
- 主键：crsp_fundno+month；wficn+month
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：03
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import polars as pl
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.runtime_paths import output_path, upstream_path
from src.schema_utils import assert_unique

panel=pl.read_parquet(upstream_path(ROOT,RUN_MODE,"03_build_fund_shareclass_month_panel","data/interim/03_fund_panel/shareclass_month_base.parquet"))
rules=CONFIG["fund_filter"]
patterns=[p for group in rules["fund_name"]["explicit_fail_patterns"].values() for p in group]
name_regex="(?i)("+"|".join(patterns)+")"
style=pl.col("crsp_obj_cd").str.to_uppercase()
fail_style_non_ed=style.is_not_null() & ~style.str.starts_with("ED")
fail_style_eds=style.str.starts_with("EDS").fill_null(False)
fail_index=pl.col("index_fund_flag").is_in(rules["index_fund"]["explicit_fail_values"]).fill_null(False)
fail_et=pl.col("et_flag").is_in(rules["etf_etn"]["explicit_fail_values"]).fill_null(False)
fail_name=pl.col("fund_name").str.contains(name_regex).fill_null(False)
explicit=fail_style_non_ed|fail_style_eds|fail_index|fail_et|fail_name
pass_condition=style.str.starts_with("ED").fill_null(False)&~fail_style_eds&~explicit
eligible=panel.with_columns(
    fail_style_non_ed.alias("fail_style_non_ed"),fail_style_eds.alias("fail_style_eds"),
    fail_index.alias("fail_index_flag"),fail_et.alias("fail_et_flag"),fail_name.alias("fail_name_pattern"),
).with_columns(
    pl.when(explicit).then(pl.lit("explicit_fail")).when(pass_condition).then(pl.lit("pass")).otherwise(pl.lit("unknown")).alias("eligibility_status")
)
month_status=eligible.group_by(["wficn","month"]).agg(
    (pl.col("eligibility_status")=="pass").sum().alias("n_pass_classes"),
    (pl.col("eligibility_status")=="unknown").sum().alias("n_unknown_classes"),
    (pl.col("eligibility_status")=="explicit_fail").sum().alias("n_explicit_fail_classes"),
    pl.col("crsp_obj_cd").filter(pl.col("eligibility_status")=="pass").drop_nulls().n_unique().alias("n_pass_styles"),
).with_columns((pl.col("n_pass_classes")>=1).alias("eligible"))
eligible=eligible.join(month_status,on=["wficn","month"],how="left")
universe=month_status.filter(pl.col("eligible"))
assert_unique(eligible,["crsp_fundno","month"],"shareclass eligibility")
assert_unique(universe,["wficn","month"],"eligible wficn month")

out1=output_path(ROOT,RUN_MODE,"04_filter_fund_universe","data/interim/04_fund_filter/shareclass_eligibility.parquet")
out2=output_path(ROOT,RUN_MODE,"04_filter_fund_universe","data/interim/04_fund_filter/eligible_wficn_months.parquet")
audit1=output_path(ROOT,RUN_MODE,"04_filter_fund_universe","data/audit/04_filter_reason_counts.csv")
audit2=output_path(ROOT,RUN_MODE,"04_filter_fund_universe","data/audit/04_style_conflicts.csv")
write_parquet_atomic(eligible,out1); write_parquet_atomic(universe,out2)
write_csv_atomic(eligible.group_by("eligibility_status").len(),audit1)
write_csv_atomic(universe.filter(pl.col("n_pass_styles")>1).with_columns(pl.lit("multiple_pass_styles").alias("reason_code")),audit2)
INPUTS_USED=[str(upstream_path(ROOT,RUN_MODE,"03_build_fund_shareclass_month_panel","data/interim/03_fund_panel/shareclass_month_base.parquet").relative_to(ROOT))]
OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out1,out2,audit1,audit2]]
INPUT_COUNTS={"panel":panel.height}; OUTPUT_COUNTS={"shareclass_status":eligible.height,"eligible_wficn_month":universe.height}
PRIMARY_KEY=["crsp_fundno","month"]; DUPLICATE_KEY_COUNT=0; EXCLUDED_ROW_COUNT=month_status.filter(~pl.col("eligible")).height
EXCEPTION_FILE=str(audit2.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["explicit_fail precedence enforced; pass+unknown aggregation locked."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "04_filter_fund_universe", stage="04", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
